In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive/medical-imaging-lakehouse'))

Mounted at /content/drive
['colab_export.zip']


In [2]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/medical-imaging-lakehouse/colab_export.zip'
extract_path = '/content/data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['gold_model_ready_export.csv', 'silver']


In [3]:
import pandas as pd

df = pd.read_csv('/content/data/gold_model_ready_export.csv')
print(df.shape)
print(df.head())

image_dir = '/content/data/silver/images_processed'
print(f"Number of images: {len(os.listdir(image_dir))}")


(3000, 3)
                      original_filename  label  split
0  09236f60-57e0-48b3-aede-a885f4dfe1b1      0  train
1  0bcb73ad-aff9-4367-adfe-f9e87b229523      0  train
2  1854a99c-bfc7-4f4e-8396-c97eebb20106      0  train
3  4d476ccd-5814-45a6-85f2-e419c9881065      0  train
4  75f0f18a-0756-4f7c-8e37-e91f9e5a24be      0  train
Number of images: 3000


In [4]:
!pip install -q timm mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.image_dir}/{row['original_filename']}.png"
        image = Image.open(img_path).convert("RGB")  # EfficientNet expects 3 channels

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

train_dataset = ChestXrayDataset(train_df, image_dir, transform=transform)
val_dataset = ChestXrayDataset(val_df, image_dir, transform=transform)
test_dataset = ChestXrayDataset(test_df, image_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

Train batches: 67, Val batches: 15, Test batches: 14


In [6]:
images, labels = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"Label values: {labels[:10]}")

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Label values: tensor([0., 1., 0., 0., 0., 0., 0., 1., 0., 0.])


In [7]:
import timm
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1)
model = model.to(device)

print(model.classifier)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Linear(in_features=1280, out_features=1, bias=True)


In [8]:
# Phase 1: freeze all layers except the classifier head
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / Total: {total_params:,}")

Trainable: 1,281 / Total: 4,008,829


In [9]:
import torch.optim as optim

pos_weight = torch.tensor([2383 / 617]).to(device)
print(f"pos_weight: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

pos_weight: 3.86


In [10]:
import mlflow
from sklearn.metrics import roc_auc_score

mlflow.set_experiment("pneumonia_classifier")

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    auroc = roc_auc_score(all_labels, all_preds)
    return total_loss / len(loader), auroc


with mlflow.start_run(run_name="phase1_frozen_backbone"):
    mlflow.log_param("phase", "1_frozen_backbone")
    mlflow.log_param("lr", 1e-3)
    mlflow.log_param("pos_weight", pos_weight.item())

    for epoch in range(3):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_auroc = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch+1}/3 - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - val_auroc: {val_auroc:.4f}")
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_auroc", val_auroc, step=epoch)

2026/06/21 09:51:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/21 09:51:38 INFO mlflow.store.db.utils: Updating database tables
2026/06/21 09:51:40 INFO mlflow.tracking.fluent: Experiment with name 'pneumonia_classifier' does not exist. Creating a new experiment.


Epoch 1/3 - train_loss: 3.4480 - val_loss: 3.3141 - val_auroc: 0.5411
Epoch 2/3 - train_loss: 2.9065 - val_loss: 2.9235 - val_auroc: 0.5915
Epoch 3/3 - train_loss: 2.5938 - val_loss: 2.8085 - val_auroc: 0.6171


In [11]:
# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable_params:,} / Total: {total_params:,}")

# Lower learning rate for fine-tuning, with cosine annealing
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)


Trainable: 4,008,829 / Total: 4,008,829


In [12]:
best_auroc = 0
best_model_path = "/content/best_model.pt"

with mlflow.start_run(run_name="phase2_fine_tune"):
    mlflow.log_param("phase", "2_fine_tune")
    mlflow.log_param("lr", 1e-4)
    mlflow.log_param("scheduler", "cosine_annealing")
    mlflow.log_param("pos_weight", pos_weight.item())

    for epoch in range(8):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_auroc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch+1}/8 - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - val_auroc: {val_auroc:.4f}")
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_auroc", val_auroc, step=epoch)

        if val_auroc > best_auroc:
            best_auroc = val_auroc
            torch.save(model.state_dict(), best_model_path)
            mlflow.log_metric("best_val_auroc", best_auroc, step=epoch)

print(f"\nBest validation AUROC: {best_auroc:.4f}")

Epoch 1/8 - train_loss: 2.0437 - val_loss: 1.9931 - val_auroc: 0.7369
Epoch 2/8 - train_loss: 0.3705 - val_loss: 2.1192 - val_auroc: 0.7192
Epoch 3/8 - train_loss: 0.1081 - val_loss: 2.2636 - val_auroc: 0.7287
Epoch 4/8 - train_loss: 0.0537 - val_loss: 2.1824 - val_auroc: 0.7461
Epoch 5/8 - train_loss: 0.0323 - val_loss: 2.3068 - val_auroc: 0.7546
Epoch 6/8 - train_loss: 0.0188 - val_loss: 2.2483 - val_auroc: 0.7480
Epoch 7/8 - train_loss: 0.0337 - val_loss: 2.3833 - val_auroc: 0.7542
Epoch 8/8 - train_loss: 0.0171 - val_loss: 2.4149 - val_auroc: 0.7498

Best validation AUROC: 0.7546


In [13]:
model.load_state_dict(torch.load(best_model_path))
test_loss, test_auroc = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}")
print(f"Test AUROC: {test_auroc:.4f}")

Test loss: 1.9692
Test AUROC: 0.7813


In [16]:
with mlflow.start_run(run_name="final_test_evaluation"):
    mlflow.log_param("training_set_size", len(train_df))
    mlflow.log_param("note", "Trained on 3000/15659 images due to disk constraints in dev env")
    mlflow.log_metric("test_auroc", test_auroc)
    mlflow.log_metric("test_loss", test_loss)

    mlflow.pytorch.log_model(
        model,
        name="model",
        serialization_format="pickle",
    )

print("Logged final model and test metrics to MLflow")

2026/06/21 09:56:58 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/21 09:56:58 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Logged final model and test metrics to MLflow


In [17]:
import os
print(os.path.exists(best_model_path))
print(f"Model file size: {os.path.getsize(best_model_path) / (1024*1024):.1f} MB")

True
Model file size: 15.6 MB


In [18]:
import shutil
drive_model_path = "/content/drive/MyDrive/medical-imaging-lakehouse/best_model.pt"
shutil.copy(best_model_path, drive_model_path)
print(f"Saved to {drive_model_path}")

Saved to /content/drive/MyDrive/medical-imaging-lakehouse/best_model.pt


In [19]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze(1)
        preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["Normal/Other", "Pneumonia"]))

Confusion Matrix:
[[280  56]
 [ 45  42]]

Classification Report:
              precision    recall  f1-score   support

Normal/Other       0.86      0.83      0.85       336
   Pneumonia       0.43      0.48      0.45        87

    accuracy                           0.76       423
   macro avg       0.65      0.66      0.65       423
weighted avg       0.77      0.76      0.77       423



In [20]:
from google.colab import files
files.download('/content/best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>